# Fine-Tuning XLM-RoBERTa for Multilingual NER

*Chapter 4 - Multilingual Named Entity Recognition · Natural Language Processing with Transformers (Tunstall, von Werra and Wolf)*

## Overview

This is the main training notebook of the chapter. We take the XLM-R token classification model from `create_model.ipynb` and fine-tune it on the PAN-X data from `tokenizing_text.ipynb`. The central question is a practical one: *if our labeled data is mostly in one language, how should we train a model that must work in several languages?*

We try four strategies and compare them with the same metric (entity-level F1 from `seqeval`):

| Strategy | Training data | Trained model folder |
|---|---|---|
| German only | German corpus | `xlm-roberta-base-finetuned-panx-de` |
| German + French | Two corpora joined | `xlm-roberta-base-finetuned-panx-de-fr` |
| One model per language | Each of `fr`, `it`, `en` separately | `xlm-roberta-base-finetuned-panx-fr`, `-it`, `-en` |
| All languages | All four corpora joined | `xlm-roberta-base-finetuned-panx-all` |

## Objective

- Configure the Hugging Face `Trainer` for token classification.
- Fine-tune on German and test the model on a German sentence.
- Measure what happens when the German model is used on other languages (zero-shot transfer).
- Train on several languages and compare the strategies in one F1 table.

## Workflow

| Step | What happens | Section |
|---|---|---|
| 1 | Import objects from the earlier notebooks and free GPU memory | Section 1 |
| 2 | Fine-tune on German and look at the training log | Section 2 |
| 3 | Record the zero-shot scores of the German model | Section 3 |
| 4 | Fine-tune on German + French and test on all languages | Section 4 |
| 5 | Fine-tune one model for each of French, Italian and English | Section 5 |
| 6 | Fine-tune on all languages together | Section 6 |
| 7 | Compare all strategies in one table | Section 7 |

## Tools and inputs

- **`transformers`**: `Trainer`, `TrainingArguments` and `DataCollatorForTokenClassification`.
- **`datasets`**: `concatenate_datasets` and `DatasetDict` to join corpora.
- **`seqeval`**: entity-level F1 (through `compute_metrics`).
- **`huggingface_hub`**: login and upload of the German model.
- **From earlier notebooks:** `langs`, `panx_ch`, `xlmr_tokenizer`, `tags`, `XLMRobertaForTokenClassification`, `tag_text`, `xlmr_config`, `device`, `encode_panx_dataset`, `panx_de_encoded` and `align_predictions`.

## Notes for the reader

- **Training time.** The stored logs show about 45 minutes for the German model and more than 45 minutes for the all-languages model. A GPU is needed.
- **Large output folders.** Each training run writes an output folder. These folders can be very large, because a checkpoint holds the model weights and the optimizer state. They are not needed to read this notebook. If you delete them, you can create them again by running the training cells. Later notebooks that load `xlm-roberta-base-finetuned-panx-de` need this folder (or the Hub copy) to run.
- **Credentials.** The notebook asks for a Hugging Face access token in a login widget. Never write the token in a code cell.
- **Shared state.** The cells change one global object, `training_args`, again and again. Run the cells in the order shown.

## Section 1 - Setup and Memory Cleanup

### 1.1 Imports

The first cell imports the libraries and brings objects from the three earlier notebooks with `import_ipynb`. This runs those notebooks, so the first run takes some time. The output is hidden with `%%capture`.

### 1.2 Freeing GPU Memory

When `create_model.ipynb` was imported, it also built a model and ran a test forward pass, which left tensors on the GPU. The next cell removes `outputs` and `xlmr_model` from that module, calls the garbage collector and empties the CUDA cache. The stored result is **0.0079 GB** of allocated GPU memory (about 8 MB), so almost all memory is free for training.

In [1]:
%%capture

import os
os.environ["USE_TF"] = "0"

import pandas as pd

from datasets import concatenate_datasets, DatasetDict
from transformers import Trainer
from transformers import TrainingArguments

import import_ipynb
from a_glance_at_dataset_and_tokenizer import langs, panx_ch, xlmr_tokenizer, tags
from create_model import (
    XLMRobertaForTokenClassification,
    tag_text,
    xlmr_config,
    device,
)
from tokenizing_text import encode_panx_dataset, panx_de_encoded, xlmr_tokenizer
from performance_measures import align_predictions

In [2]:
import gc
import torch
import create_model

# Importing create_model ran that notebook, which left a model and tensors in memory.
# We delete them to free GPU memory before training.
if hasattr(create_model, "outputs"):
    del create_model.outputs

if hasattr(create_model, "xlmr_model"):
    del create_model.xlmr_model

# Release Python objects and the cached GPU memory
gc.collect()
torch.cuda.empty_cache()

print(
    "allocated:",
    torch.cuda.memory_allocated() / 1024**3,
    "GB"
)

allocated: 0.007935047149658203 GB


## Section 2 - Fine-Tuning on German

### 2.1 Hyperparameters and Training Arguments

The values below are set in the next two cells. Everything that is not listed (for example the learning rate) keeps the default value of the library.

| Setting | Value | Meaning |
|---|---|---|
| `num_epochs` | 3 | Number of passes over the training data |
| `batch_size` | 24 | Training batch size (the code also uses it for `per_gpu_eval_batch_size`) |
| `logging_steps` | `len(train) // batch_size` = 524 | Log the loss about once per epoch (derived: 12,580 // 24) |
| `eval_strategy` | `"epoch"` | Evaluate on the validation split after each epoch |
| `weight_decay` | 0.01 | Regularization of the weights |
| `save_steps` | `1e6` | Very large, so no checkpoint is saved during training |
| `push_to_hub` | `True` | Upload the model after training |
| `log_level`, `disable_tqdm` | `"error"`, `False` | Show only errors in the log, and keep the progress bars |
| `output_dir` | `xlm-roberta-base-finetuned-panx-de` | Folder and default repository name |

In [3]:
num_epochs = 3
batch_size = 24
# Log the training loss about once per epoch
logging_steps = len(panx_de_encoded["train"]) // batch_size
xlmr_model_name = "xlm-roberta-base"
# Name of the output folder (it is also the default name of the Hub repository)
model_name = f"{xlmr_model_name}-finetuned-panx-de"

In [4]:
# eval_strategy="epoch": evaluate after every epoch.
# save_steps=1e6: a very large value, so no checkpoint is saved during training.
training_args = TrainingArguments(
    output_dir=model_name, log_level="error", num_train_epochs=num_epochs,
    per_device_train_batch_size=batch_size,
    per_gpu_eval_batch_size=batch_size, eval_strategy="epoch",
    save_steps=1e6, weight_decay=0.01, disable_tqdm=False,
    logging_steps=logging_steps, push_to_hub=True,
)

### 2.2 Logging in to the Hugging Face Hub

`notebook_login()` shows a widget in which you paste your own access token. It is needed because `push_to_hub=True` uploads the trained model. The stored output is only the empty login widget, and no token is saved in the notebook.

In [4]:
from huggingface_hub import notebook_login

notebook_login()

### 2.3 Metric, Data Collator and Model Initialization

Three small pieces are needed before we can create a `Trainer`:

- **`compute_metrics`** receives the logits and labels at the end of every evaluation. It calls `align_predictions` to remove the `-100` positions and returns the entity-level F1 score.
- **`DataCollatorForTokenClassification`** builds the batches. It pads the input ids and the attention masks, and it pads the labels with `-100`, so the padding is ignored by the loss.
- **`model_init`** creates a fresh model. When we give a `Trainer` a `model_init` function and not a model object, every call of `train()` starts from the pretrained weights. This is important, because we train many models in this notebook.

In [5]:
from seqeval.metrics import f1_score

def compute_metrics(eval_pred):
    """Compute the entity-level F1 score for the Trainer.

    Args:
        eval_pred: `EvalPrediction` that holds the model `predictions` (logits)
            and the `label_ids`.

    Returns:
        A dictionary with one key, `"f1"`, that holds the `seqeval` F1 score.
    """
    # align_predictions returns the predicted tags first and the true tags second
    y_pred, y_true = align_predictions(eval_pred.predictions, eval_pred.label_ids)

    return {"f1": f1_score(y_true, y_pred)}

In [6]:
from transformers import DataCollatorForTokenClassification

# Pads input ids, attention masks and labels (with -100) to the longest sequence in a batch
data_collator = DataCollatorForTokenClassification(xlmr_tokenizer)

In [7]:
def model_init():
    """Create a new XLM-R token classification model.

    The Trainer calls this function at the start of every `train()` call, so each
    training run begins with the pretrained body and a new random head.

    Returns:
        An `XLMRobertaForTokenClassification` model on the selected device.

    Notes:
        Uses the global variables `xlmr_model_name`, `xlmr_config` and `device`.
    """
    return (XLMRobertaForTokenClassification
           .from_pretrained(xlmr_model_name, config=xlmr_config)
           .to(device))

### 2.4 Creating the Trainer and Training

The `Trainer` combines the model, the arguments, the data collator, the metric function and the data. It trains on `panx_de_encoded["train"]` and evaluates on `panx_de_encoded["validation"]`. The tokenizer is given as `processing_class`, so it is saved together with the model.

In [8]:
# model_init (instead of model) creates a fresh model each time train() is called
trainer = Trainer(model_init=model_init, args=training_args,
                  data_collator=data_collator, compute_metrics=compute_metrics,
                  train_dataset=panx_de_encoded["train"],
                  eval_dataset=panx_de_encoded["validation"],
                  processing_class=xlmr_tokenizer)

The stored training table shows 1,575 steps (3 epochs) and a run time of 45 minutes and 15 seconds. The numbers are from the validation split:

| Epoch | Training loss | Validation loss | Validation F1 |
|---|---|---|---|
| 1 | 0.2646 | 0.1563 | 0.8278 |
| 2 | 0.1283 | 0.1357 | 0.8540 |
| 3 | 0.0813 | 0.1355 | 0.8622 |

The training loss falls in every epoch, the validation loss falls more slowly and is almost the same in epochs 2 and 3, and the F1 score keeps rising. The log lines at the top of the output come from the experiment tracking libraries (Alembic and MLflow) in the environment, and they are not errors.

In [ ]:
# Train for 3 epochs, then upload the model to the Hugging Face Hub
trainer.train()
trainer.push_to_hub(commit_message="Training Completed!")


2026/09/09 12:09:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/09/09 12:09:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/09/09 12:09:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/09/09 12:09:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/09/09 12:09:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/09/09 12:09:41 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/09/09 12:09:42 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/09/09 12:09:42 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Epoch,Training Loss,Validation Loss,F1
1,0.264600,0.156261,0.827838
2,0.128300,0.135702,0.853998
3,0.081300,0.135523,0.862184


### 2.5 Trying the Fine-Tuned Model

The next cell loads the model from the output folder. The second cell tags a German sentence with `tag_text`. The stored result shows that the model found all three entity types: `Jeff Dean` as a person (`B-PER`, `I-PER`), `Google` as an organization (`B-ORG`) and `Kalifornien` as a location (`B-LOC`, `I-LOC`).

> **Note:** This step needs the folder `xlm-roberta-base-finetuned-panx-de` on disk.

In [10]:
# Load the fine-tuned German model from its local output folder
trained_model = XLMRobertaForTokenClassification.from_pretrained(
    model_name
).to(device)

In [11]:
text_de = "Jeff Dean ist ein Informatiker bei Google in Kalifornien"
tag_text(text_de, tags, trained_model, xlmr_tokenizer)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
Tokens,<s>,▁Jeff,▁De,an,▁ist,▁ein,▁Informati,ker,▁bei,▁Google,▁in,▁Kaliforni,en,</s>
Tags,O,B-PER,I-PER,I-PER,O,O,O,O,O,B-ORG,O,B-LOC,I-LOC,O


## Section 3 - Zero-Shot Reference Scores

The German-only model is also tested on the other three languages. This is **zero-shot cross-lingual transfer**: the model never saw French, Italian or English labels. The scores are measured in `cross_lingual_transfer.ipynb`, and the four values typed into the next cell match the test scores printed there. They are entered by hand, so this cell does not compute anything.

| Test language | `de` | `fr` | `it` | `en` |
|---|---|---|---|---|
| F1 of the German-only model | 0.866 | 0.690 | 0.688 | 0.582 |

In [10]:
from collections import defaultdict

# Test F1 scores of the German-only model, entered by hand (not computed in this notebook)
f1_scores = defaultdict(dict)
f1_scores["de"]["de"] = 0.866
f1_scores["de"]["fr"] = 0.690
f1_scores["de"]["it"] = 0.688
f1_scores["de"]["en"] = 0.582

## Section 4 - Training on Several Languages

### 4.1 Joining Corpora

`concatenate_splits` joins several `DatasetDict` objects split by split and shuffles the result with `seed=42`. The shuffle mixes the languages inside each batch. The next cell encodes the French corpus and joins it with the German one. The new training set has 12,580 + 4,580 = **17,160** examples (derived from the sizes of the first notebook).

In [11]:
def concatenate_splits(corpora):
    """Join several corpora split by split and shuffle the result.

    Args:
        corpora: List of `DatasetDict` objects that have the same split names.

    Returns:
        A `DatasetDict` in which each split is the concatenation of that split
        from all corpora, shuffled with `seed=42`.
    """
    multi_corpus = DatasetDict()
    for split in corpora[0].keys():
        multi_corpus[split] = concatenate_datasets(
            [corpus[split] for corpus in corpora]
        ).shuffle(seed=42)
    return multi_corpus

In [12]:
# Encode the French corpus and join it with the German one
panx_fr_encoded = encode_panx_dataset(panx_ch['fr'])
panx_de_fr_encoded = concatenate_splits([panx_de_encoded, panx_fr_encoded])

### 4.2 Training the German + French Model

We change three values of the same `training_args` object: the logging interval for the new data size, `push_to_hub=False` (this model is not uploaded) and a new output folder. Then we create a new `Trainer` and train it.

In [13]:
# Re-use the same TrainingArguments object with new values for the de-fr run
training_args.logging_steps = len(panx_de_fr_encoded["train"]) // batch_size
training_args.push_to_hub = False
training_args.output_dir = "xlm-roberta-base-finetuned-panx-de-fr"

In [14]:
trainer = Trainer(
    model_init=model_init, args=training_args, processing_class=xlmr_tokenizer,
    data_collator=data_collator, compute_metrics=compute_metrics,
    train_dataset=panx_de_fr_encoded["train"], eval_dataset=panx_de_fr_encoded["validation"]
)

The stored table shows 2,145 steps and a run time of 37 minutes and 41 seconds. The validation scores are:

| Epoch | Training loss | Validation loss | Validation F1 |
|---|---|---|---|
| 1 | 0.2805 | 0.1773 | 0.8222 |
| 2 | 0.1464 | 0.1660 | 0.8498 |
| 3 | 0.0939 | 0.1595 | 0.8565 |

The result of `trainer.train()` also shows a training loss of 0.1736 for the whole run.

In [14]:
trainer.train()

2026/10/14 10:43:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/14 10:43:11 INFO mlflow.store.db.utils: Updating database tables


Epoch,Training Loss,Validation Loss,F1
1,0.280500,0.177282,0.822229
2,0.146400,0.165992,0.849823
3,0.093900,0.159498,0.856500


TrainOutput(global_step=2145, training_loss=0.17359836818454982, metrics={'train_runtime': 2266.626, 'train_samples_per_second': 22.712, 'train_steps_per_second': 0.946, 'total_flos': 1141065108882864.0, 'train_loss': 0.17359836818454982, 'epoch': 3.0})

### 4.3 Evaluating on Every Language

Two small helpers make the evaluation short. `get_f1_score` runs `trainer.predict` on a dataset and returns the F1 score. `evaluate_lang_performance` encodes the corpus of one language first and then calls `get_f1_score` on its test split.

In [16]:
def get_f1_score(trainer, dataset):
    """Predict on a dataset and return the entity-level F1 score.

    Args:
        trainer: A `Trainer` that has a `compute_metrics` function.
        dataset: Encoded dataset with labels (usually a test split).

    Returns:
        The F1 score as a float. The key is `test_f1` because `Trainer.predict`
        adds the prefix `test_` to the metric names.
    """
    return trainer.predict(dataset).metrics["test_f1"]

In [17]:
def evaluate_lang_performance(lang, trainer):
    """Encode the corpus of one language and return its test F1 score.

    Args:
        lang: Language code of a corpus in `panx_ch` (for example `"fr"`).
        trainer: Trainer that holds the model to evaluate.

    Returns:
        The F1 score on the test split of the language.
    """
    panx_ds = encode_panx_dataset(panx_ch[lang])
    return get_f1_score(trainer, panx_ds["test"])

The loop below tests the German + French model on the test split of all four languages. The stored scores are:

| Test language | `de` | `fr` | `it` | `en` |
|---|---|---|---|---|
| German-only model (Section 3) | 0.866 | 0.690 | 0.688 | 0.582 |
| German + French model | 0.869 | 0.862 | 0.795 | 0.653 |

Adding French helps most on French (0.862 against 0.690), and it also helps Italian (+0.107) and English (+0.071). German stays almost the same (+0.003). The differences are calculated from the rounded values above.

In [19]:
for lang in langs:
    f1 = evaluate_lang_performance(lang, trainer)
    print(f"F1-Score of [de-fr] model on [{lang}] dataset: {f1:.3f}")

F1-Score of [de-fr] model on [de] dataset: 0.869


F1-Score of [de-fr] model on [fr] dataset: 0.862


F1-Score of [de-fr] model on [it] dataset: 0.795


F1-Score of [de-fr] model on [en] dataset: 0.653


## Section 5 - Fine-Tuning One Model for Each Language

The function `train_on_subset` trains a new model on a chosen number of training examples, evaluates it on the test split and returns the result as a one-row table. In this section, we use all training examples of each language. (A copy of this function is used with smaller numbers in `cross_lingual_transfer.ipynb`.)

In [18]:
def train_on_subset(dataset, num_samples):
    """Fine-tune a new model on a part of the training data and test it.

    Args:
        dataset: Encoded `DatasetDict` with `train`, `validation` and `test`
            splits.
        num_samples: Number of training examples to use. They are chosen after
            a shuffle with `seed=42`.

    Returns:
        A `pandas.DataFrame` with one row and the columns `num_samples` and
        `f1_score` (the test F1 score of the trained model).

    Notes:
        Uses the globals `training_args`, `model_init`, `data_collator`,
        `compute_metrics` and `batch_size`, and it changes
        `training_args.logging_steps`. The model is pushed to the Hub only if
        `training_args.push_to_hub` is true.
    """
    train_ds = dataset['train'].shuffle(seed=42).select(range(num_samples))
    val_ds = dataset['validation']
    test_ds = dataset['test']
    # Log about once per epoch for this size of training set
    training_args.logging_steps = len(train_ds) // batch_size
    trainer = Trainer(model_init=model_init, args=training_args,
        data_collator=data_collator, compute_metrics=compute_metrics, 
        train_dataset=train_ds, eval_dataset=val_ds, processing_class=xlmr_tokenizer,
    )
    trainer.train()

    if training_args.push_to_hub:
        trainer.push_to_hub(commit_message="Training Completed!")

    # Evaluate the trained model on the test split
    f1_score = get_f1_score(trainer, test_ds)
    return pd.DataFrame.from_dict(
        {"num_samples": [len(train_ds)], "f1_score": [f1_score]}
    )

The loop trains a separate model for French, Italian and English. Each model has its own output folder, and its F1 score is saved in `f1_scores`. The monolingual corpora are also collected in the list `corpora` for the next section. The stored training tables show these validation scores after epoch 3:

| Language | Training examples | Steps | Validation F1 (epoch 3) |
|---|---|---|---|
| `fr` | 4,580 | 573 | 0.8399 |
| `it` | 1,680 | 210 | 0.8230 |
| `en` | 1,180 | 150 | 0.7023 |

The smaller the training set, the lower the validation score. English has the fewest examples and the lowest score.

In [19]:
corpora = [panx_de_encoded]
# Exclude German from iteration
for lang in langs[1:]:
    training_args.output_dir = f"xlm-roberta-base-finetuned-panx-{lang}"
    # Fine-tune on monolingual corpus
    ds_encoded = encode_panx_dataset(panx_ch[lang])
    metrics = train_on_subset(ds_encoded, ds_encoded["train"].num_rows)
    # Collect F1-scores in common dict
    f1_scores[lang][lang] = metrics["f1_score"][0]
    # Add monolingual corpus to list of corpora to concatenate
    corpora.append(ds_encoded)

Epoch,Training Loss,Validation Loss,F1
1,0.542000,0.315651,0.807333
2,0.259600,0.276366,0.824535
3,0.170800,0.279950,0.839926


Epoch,Training Loss,Validation Loss,F1
1,0.703300,0.308881,0.776746
2,0.276300,0.265506,0.785973
3,0.184300,0.268424,0.822998


Epoch,Training Loss,Validation Loss,F1
1,1.007400,0.532548,0.564378
2,0.493900,0.419557,0.656733
3,0.381900,0.386594,0.702282


## Section 6 - Fine-Tuning on All Languages

The last strategy joins the German corpus with the three other corpora and trains one model on everything. The training set has 12,580 + 4,580 + 1,680 + 1,180 = **20,020** examples. About 63% of them are German (derived). The stored training table shows 2,505 steps and a run time of 46 minutes and 24 seconds:

| Epoch | Training loss | Validation loss | Validation F1 |
|---|---|---|---|
| 1 | 0.2945 | 0.2023 | 0.8096 |
| 2 | 0.1566 | 0.1646 | 0.8464 |
| 3 | 0.0993 | 0.1758 | 0.8522 |

In epoch 3 the validation loss goes up a little (0.1646 to 0.1758) while the F1 score still rises. This is a small change, and we do not draw a conclusion from it.

In [21]:
# Train one model on the concatenation of the four corpora
corpora_encoded = concatenate_splits(corpora)
training_args.logging_steps = len(corpora_encoded["train"]) // batch_size
training_args.output_dir = "xlm-roberta-base-finetuned-panx-all"
trainer = Trainer(
    model_init=model_init, args=training_args, compute_metrics=compute_metrics,
    processing_class=xlmr_tokenizer, data_collator=data_collator,
    train_dataset=corpora_encoded["train"], eval_dataset=corpora_encoded["validation"]
)
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.294500,0.202265,0.809603
2,0.156600,0.164600,0.846352
3,0.099300,0.175798,0.852166


TrainOutput(global_step=2505, training_loss=0.18328902220833088, metrics={'train_runtime': 2784.736, 'train_samples_per_second': 21.568, 'train_steps_per_second': 0.9, 'total_flos': 1314051381280176.0, 'train_loss': 0.18328902220833088, 'epoch': 3.0})

## Section 7 - Comparing the Strategies

The last cell collects all test scores in one table. The rows show how the model was trained, and the columns show the language of the test data:

- `de`: the German-only model (typed in by hand in Section 3).
- `each`: the model trained on the same language as the test data (the diagonal of the monolingual results).
- `all`: the model trained on all four languages.

In [24]:
# Test F1 of the all-languages model on the test split of each language
for idx, lang in enumerate(langs):
    f1_scores["all"][lang] = get_f1_score(trainer, corpora[idx]["test"])

scores_data = {
    "de": f1_scores["de"],
    "each": {lang: f1_scores[lang][lang] for lang in langs},
    "all": f1_scores["all"],
}
# Rows: training data of the model. Columns: language of the test data.
f1_scores_df = pd.DataFrame(scores_data).T.round(4)
f1_scores_df.rename_axis(index="Fine-tuned on", columns="Evaluated on", inplace=True)
f1_scores_df

Evaluated on,de,fr,it,en
Fine-tuned on,,,,
de,0.8660,0.6900,0.6880,0.5820
each,0.8660,0.8405,0.8186,0.6841
all,0.8664,0.8642,0.8620,0.7703


## Conclusion

We fine-tuned XLM-R for NER with four strategies and tested every model on the test split of each language. All numbers below are read from the stored outputs.

### Training runs

| Run | Training examples | Steps | Time | Validation F1 (epoch 3) |
|---|---|---|---|---|
| German only | 12,580 | 1,575 | 45:15 | 0.8622 |
| German + French | 17,160 | 2,145 | 37:41 | 0.8565 |
| French only | 4,580 | 573 | 09:24 | 0.8399 |
| Italian only | 1,680 | 210 | 08:01 | 0.8230 |
| English only | 1,180 | 150 | 05:25 | 0.7023 |
| All languages | 20,020 | 2,505 | 46:24 | 0.8522 |

### Test F1 by training strategy (rows) and test language (columns)

| Fine-tuned on | `de` | `fr` | `it` | `en` |
|---|---|---|---|---|
| `de` (German only) | 0.8660 | 0.6900 | 0.6880 | 0.5820 |
| `each` (same language) | 0.8660 | 0.8405 | 0.8186 | 0.6841 |
| `all` (all languages) | 0.8664 | 0.8642 | 0.8620 | 0.7703 |

### Interpretation

- **Zero-shot transfer works, but it loses a lot.** The German model reaches 0.866 on German but only 0.582 to 0.690 on the other languages.
- **Training on all languages is the best strategy here.** It is equal or better than the monolingual model on every language. The gain over `each` is +0.0004 for `de`, +0.0237 for `fr`, +0.0434 for `it` and +0.0862 for `en`.
- **The languages with less data gain most.** English has the smallest corpus, and it gains the most.
- **The German gain is too small to matter.** Each setting was trained once, so we cannot say that a difference of 0.0004 is real.

### Limits and next steps

- The validation F1 of the all-languages model (0.8522) is lower than that of the German-only model (0.8622). These two numbers are not directly comparable, because the validation sets are different (the all-languages set contains four languages). Only the test table above compares the models on the same test data.
- Each result comes from one training run. Several seeds would show how stable the differences are.
- The next notebooks study the errors of the German model (`error_analysis.ipynb`) and the learning curve for French (`cross_lingual_transfer.ipynb`).

## **Resources & References**

**Official Documentation**

- [Hugging Face Transformers: `Trainer` and `TrainingArguments`](https://huggingface.co/docs/transformers/main_classes/trainer)
- [Hugging Face Transformers: Data collators](https://huggingface.co/docs/transformers/main_classes/data_collator)
- [Hugging Face Transformers: Token classification guide](https://huggingface.co/docs/transformers/tasks/token_classification)
- [`seqeval` repository](https://github.com/chakki-works/seqeval)
- [Model card: `FacebookAI/xlm-roberta-base`](https://huggingface.co/FacebookAI/xlm-roberta-base)

**Key Papers / Research**

- Conneau, A. et al. (2019). [Unsupervised Cross-lingual Representation Learning at Scale](https://arxiv.org/abs/1911.02116).
- Hu, J. et al. (2020). [XTREME: A Massively Multilingual Multi-task Benchmark for Evaluating Cross-lingual Generalization](https://arxiv.org/abs/2003.11080).
- Pan, X. et al. (2017). [Cross-lingual Name Tagging and Linking for 282 Languages](https://aclanthology.org/P17-1178/). ACL 2017.

**Additional Reading**

- Tunstall, L., von Werra, L., Wolf, T. (2022). [Natural Language Processing with Transformers, Revised Edition](https://www.oreilly.com/library/view/natural-language-processing/9781098136789/). O'Reilly Media. Chapter 4: Multilingual Named Entity Recognition.